In [ ]:
!pip install -q transformers einops timm segment-anything scipy pandas
!pip install -q git+https://github.com/facebookresearch/sam2.git
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

In [ ]:
import os
import gc
import cv2
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from scipy.spatial.distance import directed_hausdorff

# Florence-2 flash_attn bypass — applied once here so it covers all later imports
import transformers.dynamic_module_utils as _dyn
_orig_check = _dyn.check_imports
def _patched_check(filename):
    try:
        return _orig_check(filename)
    except ImportError as e:
        if 'flash_attn' in str(e):
            return _dyn.get_relative_imports(filename)
        raise
_dyn.check_imports = _patched_check

# ================= CONFIGURATION =================
TEST_IMG_DIR   = "/kaggle/input/YOUR_TEST_DATASET/test/images"   # <--- UPDATE
TEST_MASK_DIR  = "/kaggle/input/YOUR_TEST_DATASET/test/masks"    # <--- UPDATE
B3_MODEL_PATH  = "/kaggle/input/YOUR_B3_MODEL/final_rat_model_b3_1024"  # <--- UPDATE
B5_MODEL_PATH  = "/kaggle/input/YOUR_B5_MODEL/final_rat_model_b5_512"   # <--- UPDATE
MAGNIFORMER_BCE_PATH  = "/kaggle/input/YOUR_STAGE2_MODEL/final_rat_model_stage2_bce"   # <--- UPDATE
MAGNIFORMER_DICE_PATH = "/kaggle/input/YOUR_STAGE2_MODEL/final_rat_model_stage2_dice"  # <--- UPDATE

SAM_CHECKPOINT  = "sam_vit_b_01ec64.pth"
SAM2_CHECKPOINT = "sam2_hiera_small.pt"
SAM2_CFG        = "sam2_hiera_s.yaml"
GDINO_MODEL_ID  = "IDEA-Research/grounding-dino-base"
FLORENCE2_ID    = "microsoft/Florence-2-large"

CONFIDENCE        = 0.5
BOUNDARY_DILATION = 7
BOX_THRESHOLD     = 0.35
TEXT_THRESHOLD    = 0.25
MAGNIFORMER_PADDING = 0.25

device      = "cuda" if torch.cuda.is_available() else "cpu"
all_results = {}   # populated by each model cell
# =================================================

# --- Shared metric functions ---
def calculate_iou(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    if union == 0: return 1.0 if inter == 0 else 0.0
    return inter / union

def calculate_dice(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    total = pred.sum() + gt.sum()
    if total == 0: return 1.0
    return (2. * inter) / total

def calculate_boundary_iou(pred, gt, dilation=BOUNDARY_DILATION):
    kernel = np.ones((dilation, dilation), dtype=np.uint8)
    gt_b   = cv2.morphologyEx(gt.astype(np.uint8),   cv2.MORPH_GRADIENT, kernel) > 0
    pred_b = cv2.morphologyEx(pred.astype(np.uint8), cv2.MORPH_GRADIENT, kernel) > 0
    inter  = np.logical_and(pred_b, gt_b).sum()
    union  = np.logical_or(pred_b, gt_b).sum()
    if union == 0: return 1.0 if inter == 0 else 0.0
    return inter / union

def calculate_hausdorff(pred, gt):
    pred_pts = np.argwhere(cv2.Canny((pred.astype(np.uint8) * 255), 0, 1) > 0)
    gt_pts   = np.argwhere(cv2.Canny((gt.astype(np.uint8)   * 255), 0, 1) > 0)
    if len(pred_pts) == 0 or len(gt_pts) == 0: return np.nan
    return max(directed_hausdorff(pred_pts, gt_pts)[0],
               directed_hausdorff(gt_pts, pred_pts)[0])

def score_frame(pred, gt):
    return {
        "mIoU"              : calculate_iou(pred, gt),
        "Dice"              : calculate_dice(pred, gt),
        "Boundary_IoU"      : calculate_boundary_iou(pred, gt),
        "Hausdorff_Distance": calculate_hausdorff(pred, gt),
    }

def get_gt_bbox(mask):
    coords = np.column_stack(np.where(mask > 0))
    if coords.size == 0: return None
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    return np.array([x_min, y_min, x_max, y_max])

def get_padded_square_crop(mask, image, padding=MAGNIFORMER_PADDING):
    """Stage-1 mask → padded square bbox → (y1,y2,x1,x2) crop coords."""
    coords = np.column_stack(np.where(mask > 0))
    if coords.size == 0:
        return None
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    bw, bh = x_max - x_min, y_max - y_min
    pad_w, pad_h = int(bw * padding), int(bh * padding)
    h, w = image.shape[:2]
    x_min = max(0, x_min - pad_w)
    y_min = max(0, y_min - pad_h)
    x_max = min(w, x_max + pad_w)
    y_max = min(h, y_max + pad_h)
    side = max(x_max - x_min, y_max - y_min)
    cx, cy = (x_min + x_max) // 2, (y_min + y_max) // 2
    x1 = max(0, cx - side // 2)
    y1 = max(0, cy - side // 2)
    x2 = min(w, x1 + side)
    y2 = min(h, y1 + side)
    return y1, y2, x1, x2

def print_summary(name, df):
    print(f"  mIoU: {df['mIoU'].mean():.4f} | "
          f"Dice: {df['Dice'].mean():.4f} | "
          f"B-IoU: {df['Boundary_IoU'].mean():.4f} | "
          f"HD: {df['Hausdorff_Distance'].mean():.2f}px")

# --- Load test set once ---
img_map  = {os.path.splitext(f)[0]: f for f in os.listdir(TEST_IMG_DIR)  if f.endswith(('.jpg', '.png'))}
mask_map = {os.path.splitext(f)[0]: f for f in os.listdir(TEST_MASK_DIR) if f.endswith(('.jpg', '.png'))}
COMMON_IDS = sorted(set(img_map.keys()) & set(mask_map.keys()))
print(f"Test set: {len(COMMON_IDS)} image/mask pairs | Device: {device.upper()}")

In [ ]:
# =============================================
#  MODEL 1 — SegFormer B3-1024 (fine-tuned)
# =============================================
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

def segformer_infer(model, processor, image_bgr, confidence=CONFIDENCE):
    h, w = image_bgr.shape[:2]
    side = max(h, w)
    padded = np.zeros((side, side, 3), dtype=np.uint8)
    y_off = (side - h) // 2
    x_off = (side - w) // 2
    padded[y_off:y_off + h, x_off:x_off + w] = image_bgr
    pil = Image.fromarray(cv2.cvtColor(padded, cv2.COLOR_BGR2RGB))
    inputs = processor(images=pil, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = torch.nn.functional.interpolate(
            model(**inputs).logits, size=(side, side), mode="bilinear", align_corners=False
        )
        rat_prob = torch.nn.functional.softmax(logits, dim=1)[0, 1]
    mask = (rat_prob > confidence).cpu().numpy().astype(np.uint8)
    return mask[y_off:y_off + h, x_off:x_off + w].astype(bool)

print("--- LOADING SEGFORMER B3-1024 ---")
sf_processor = SegformerImageProcessor.from_pretrained(B3_MODEL_PATH)
sf_model     = SegformerForSemanticSegmentation.from_pretrained(B3_MODEL_PATH).to(device).eval()

rows = []
for cid in tqdm(COMMON_IDS, desc="SegFormer B3-1024"):
    image = cv2.imread(os.path.join(TEST_IMG_DIR, img_map[cid]))
    gt    = np.where(cv2.imread(os.path.join(TEST_MASK_DIR, mask_map[cid]), cv2.IMREAD_GRAYSCALE) > 0, 1, 0).astype(bool)
    pred  = segformer_infer(sf_model, sf_processor, image)
    rows.append({"Frame_ID": cid, **score_frame(pred, gt)})

all_results["SegFormer B3-1024"] = pd.DataFrame(rows)
del sf_model, sf_processor
torch.cuda.empty_cache(); gc.collect()

print("SegFormer B3-1024 done.")
print_summary("SegFormer B3-1024", all_results["SegFormer B3-1024"])

In [ ]:
# =============================================
#  MODEL 2 — SegFormer B5-512 (fine-tuned)
# =============================================
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

print("--- LOADING SEGFORMER B5-512 ---")
sf_processor = SegformerImageProcessor.from_pretrained(B5_MODEL_PATH)
sf_model     = SegformerForSemanticSegmentation.from_pretrained(B5_MODEL_PATH).to(device).eval()

rows = []
for cid in tqdm(COMMON_IDS, desc="SegFormer B5-512"):
    image = cv2.imread(os.path.join(TEST_IMG_DIR, img_map[cid]))
    gt    = np.where(cv2.imread(os.path.join(TEST_MASK_DIR, mask_map[cid]), cv2.IMREAD_GRAYSCALE) > 0, 1, 0).astype(bool)
    pred  = segformer_infer(sf_model, sf_processor, image)
    rows.append({"Frame_ID": cid, **score_frame(pred, gt)})

all_results["SegFormer B5-512"] = pd.DataFrame(rows)
del sf_model, sf_processor
torch.cuda.empty_cache(); gc.collect()

print("SegFormer B5-512 done.")
print_summary("SegFormer B5-512", all_results["SegFormer B5-512"])

In [ ]:
# =============================================
#  MODEL 3 — Magniformer BCE  (B3-1024 → Stage-2 BCE specialist)
#  MODEL 4 — Magniformer Dice (B3-1024 → Stage-2 Dice specialist)
# =============================================
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

def magniformer_infer(stage1_model, stage1_proc, stage2_model, stage2_proc, image_bgr):
    h, w = image_bgr.shape[:2]

    # Stage 1: coarse full-frame prediction
    stage1_mask = segformer_infer(stage1_model, stage1_proc, image_bgr).astype(np.uint8)

    full_mask = np.zeros((h, w), dtype=np.uint8)
    crop_coords = get_padded_square_crop(stage1_mask, image_bgr)
    if crop_coords is not None:
        y1, y2, x1, x2 = crop_coords
        crop = image_bgr[y1:y2, x1:x2]
        if crop.shape[0] > 0 and crop.shape[1] > 0:
            # Stage 2: refine on the magnified crop
            crop_mask = segformer_infer(stage2_model, stage2_proc, crop).astype(np.uint8)
            full_mask[y1:y2, x1:x2] = crop_mask

    return full_mask.astype(bool)

print("--- LOADING MAGNIFORMER STAGE 1 (B3-1024) ---")
s1_proc,  s1_model  = (SegformerImageProcessor.from_pretrained(B3_MODEL_PATH),
                        SegformerForSemanticSegmentation.from_pretrained(B3_MODEL_PATH).to(device).eval())

print("--- LOADING MAGNIFORMER STAGE 2 BCE ---")
s2b_proc, s2b_model = (SegformerImageProcessor.from_pretrained(MAGNIFORMER_BCE_PATH),
                        SegformerForSemanticSegmentation.from_pretrained(MAGNIFORMER_BCE_PATH).to(device).eval())

print("--- LOADING MAGNIFORMER STAGE 2 DICE ---")
s2d_proc, s2d_model = (SegformerImageProcessor.from_pretrained(MAGNIFORMER_DICE_PATH),
                        SegformerForSemanticSegmentation.from_pretrained(MAGNIFORMER_DICE_PATH).to(device).eval())

rows_bce, rows_dice = [], []
for cid in tqdm(COMMON_IDS, desc="Magniformer"):
    image   = cv2.imread(os.path.join(TEST_IMG_DIR, img_map[cid]))
    gt_gray = cv2.imread(os.path.join(TEST_MASK_DIR, mask_map[cid]), cv2.IMREAD_GRAYSCALE)
    gt      = np.where(gt_gray > 0, 1, 0).astype(bool)

    pred_bce  = magniformer_infer(s1_model, s1_proc, s2b_model, s2b_proc, image)
    pred_dice = magniformer_infer(s1_model, s1_proc, s2d_model, s2d_proc, image)

    rows_bce.append( {"Frame_ID": cid, **score_frame(pred_bce,  gt)})
    rows_dice.append({"Frame_ID": cid, **score_frame(pred_dice, gt)})

all_results["Magniformer BCE"]  = pd.DataFrame(rows_bce)
all_results["Magniformer Dice"] = pd.DataFrame(rows_dice)
del s1_model, s1_proc, s2b_model, s2b_proc, s2d_model, s2d_proc
torch.cuda.empty_cache(); gc.collect()

print("Magniformer BCE done.")
print_summary("Magniformer BCE",  all_results["Magniformer BCE"])
print("Magniformer Dice done.")
print_summary("Magniformer Dice", all_results["Magniformer Dice"])

In [ ]:
# =============================================
#  MODEL 3 — SAM v1 (oracle GT bbox prompt)
# =============================================
from segment_anything import sam_model_registry, SamPredictor

print("--- LOADING SAM V1 ---")
sam       = sam_model_registry["vit_b"](checkpoint=SAM_CHECKPOINT).to(device)
predictor = SamPredictor(sam)

rows = []
for cid in tqdm(COMMON_IDS, desc="SAM v1"):
    image    = cv2.imread(os.path.join(TEST_IMG_DIR, img_map[cid]))
    gt_gray  = cv2.imread(os.path.join(TEST_MASK_DIR, mask_map[cid]), cv2.IMREAD_GRAYSCALE)
    gt       = np.where(gt_gray > 0, 1, 0).astype(bool)
    bbox     = get_gt_bbox(gt)
    if bbox is None:
        rows.append({"Frame_ID": cid, "mIoU": 0.0, "Dice": 0.0, "Boundary_IoU": 0.0, "Hausdorff_Distance": np.nan})
        continue
    predictor.set_image(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    masks, scores, _ = predictor.predict(
        point_coords=None, point_labels=None, box=bbox[None, :], multimask_output=True
    )
    pred = masks[np.argmax(scores)]
    rows.append({"Frame_ID": cid, **score_frame(pred, gt)})

all_results["SAM v1 (oracle)"] = pd.DataFrame(rows)
del sam, predictor
torch.cuda.empty_cache(); gc.collect()

print("SAM v1 done.")
print_summary("SAM v1 (oracle)", all_results["SAM v1 (oracle)"])

In [ ]:
# =============================================
#  MODEL 4 — SAM 2 (oracle GT bbox prompt)
# =============================================
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

print("--- LOADING SAM 2 ---")
sam2_model = build_sam2(SAM2_CFG, SAM2_CHECKPOINT, device=device)
predictor  = SAM2ImagePredictor(sam2_model)

rows = []
for cid in tqdm(COMMON_IDS, desc="SAM 2"):
    image   = cv2.imread(os.path.join(TEST_IMG_DIR, img_map[cid]))
    gt_gray = cv2.imread(os.path.join(TEST_MASK_DIR, mask_map[cid]), cv2.IMREAD_GRAYSCALE)
    gt      = np.where(gt_gray > 0, 1, 0).astype(bool)
    bbox    = get_gt_bbox(gt)
    if bbox is None:
        rows.append({"Frame_ID": cid, "mIoU": 0.0, "Dice": 0.0, "Boundary_IoU": 0.0, "Hausdorff_Distance": np.nan})
        continue
    predictor.set_image(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    masks, scores, _ = predictor.predict(
        point_coords=None, point_labels=None, box=bbox[None, :], multimask_output=True
    )
    pred = masks[np.argmax(scores)]
    rows.append({"Frame_ID": cid, **score_frame(pred, gt)})

all_results["SAM 2 (oracle)"] = pd.DataFrame(rows)
del sam2_model, predictor
torch.cuda.empty_cache(); gc.collect()

print("SAM 2 done.")
print_summary("SAM 2 (oracle)", all_results["SAM 2 (oracle)"])

In [ ]:
# =============================================
#  MODEL 5 — Grounded-SAM 2 (zero-shot)
# =============================================
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

print("--- LOADING GROUNDING DINO ---")
gd_processor = AutoProcessor.from_pretrained(GDINO_MODEL_ID)
gd_model     = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_MODEL_ID).to(device).eval()

print("--- LOADING SAM 2 ---")
sam2_model = build_sam2(SAM2_CFG, SAM2_CHECKPOINT, device=device)
sam2_pred  = SAM2ImagePredictor(sam2_model)

rows = []
for cid in tqdm(COMMON_IDS, desc="Grounded-SAM2"):
    image     = cv2.imread(os.path.join(TEST_IMG_DIR, img_map[cid]))
    gt_gray   = cv2.imread(os.path.join(TEST_MASK_DIR, mask_map[cid]), cv2.IMREAD_GRAYSCALE)
    gt        = np.where(gt_gray > 0, 1, 0).astype(bool)
    pil_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    inputs = gd_processor(images=pil_image, text="rat.", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gd_model(**inputs)
    results = gd_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
        target_sizes=[pil_image.size[::-1]]
    )[0]

    if len(results["boxes"]) == 0:
        rows.append({"Frame_ID": cid, "mIoU": 0.0, "Dice": 0.0, "Boundary_IoU": 0.0, "Hausdorff_Distance": np.nan})
        continue

    bbox = results["boxes"][torch.argmax(results["scores"])].cpu().numpy()
    sam2_pred.set_image(np.array(pil_image))
    masks, scores, _ = sam2_pred.predict(
        point_coords=None, point_labels=None, box=bbox[None, :], multimask_output=True
    )
    pred = masks[np.argmax(scores)]
    rows.append({"Frame_ID": cid, **score_frame(pred, gt)})

all_results["Grounded-SAM2"] = pd.DataFrame(rows)
del gd_model, gd_processor, sam2_model, sam2_pred
torch.cuda.empty_cache(); gc.collect()

print("Grounded-SAM2 done.")
print_summary("Grounded-SAM2", all_results["Grounded-SAM2"])

In [ ]:
# =============================================
#  MODEL 6 — Florence-2 (zero-shot)
# =============================================
from transformers import AutoProcessor, AutoModelForCausalLM

TASK_PROMPT = "<REFERRING_EXPRESSION_SEGMENTATION>"
TEXT_INPUT  = " rat"

print("--- LOADING FLORENCE-2 ---")
fl_processor = AutoProcessor.from_pretrained(FLORENCE2_ID, trust_remote_code=True)
fl_model     = AutoModelForCausalLM.from_pretrained(
    FLORENCE2_ID, torch_dtype=torch.float16, trust_remote_code=True
).to(device).eval()

rows = []
for cid in tqdm(COMMON_IDS, desc="Florence-2"):
    image   = cv2.imread(os.path.join(TEST_IMG_DIR, img_map[cid]))
    h, w    = image.shape[:2]
    gt_gray = cv2.imread(os.path.join(TEST_MASK_DIR, mask_map[cid]), cv2.IMREAD_GRAYSCALE)
    gt      = np.where(gt_gray > 0, 1, 0).astype(bool)
    pil_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    prompt = TASK_PROMPT + TEXT_INPUT
    inputs = fl_processor(text=prompt, images=pil_image, return_tensors="pt").to(device, torch.float16)
    with torch.no_grad():
        generated_ids = fl_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            do_sample=False,
            num_beams=3,
        )
    generated_text  = fl_processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed          = fl_processor.post_process_generation(generated_text, task=TASK_PROMPT, image_size=(w, h))

    pred_mask = np.zeros((h, w), dtype=np.uint8)
    polys_dict = parsed.get(TASK_PROMPT, {})
    for obj_polys in polys_dict.get('polygons', polys_dict.get('Polygons', [])):
        for poly in obj_polys:
            if len(poly) >= 6:
                cv2.fillPoly(pred_mask, [np.array(poly).reshape(-1, 2).astype(np.int32)], 1)
    pred = pred_mask.astype(bool)
    rows.append({"Frame_ID": cid, **score_frame(pred, gt)})

all_results["Florence-2"] = pd.DataFrame(rows)
del fl_model, fl_processor
torch.cuda.empty_cache(); gc.collect()

print("Florence-2 done.")
print_summary("Florence-2", all_results["Florence-2"])

In [ ]:
# =============================================
#  FINAL COMPARISON TABLE
# =============================================
summary_rows = []
for model_name, df in all_results.items():
    df.to_csv(f"/kaggle/working/{model_name.replace(' ', '_').replace('/', '-')}_metrics.csv", index=False)
    summary_rows.append({
        "Model"             : model_name,
        "N"                 : len(df),
        "mIoU"              : round(df["mIoU"].mean(), 4),
        "Dice"              : round(df["Dice"].mean(), 4),
        "Boundary_IoU"      : round(df["Boundary_IoU"].mean(), 4),
        "Hausdorff_Distance": round(df["Hausdorff_Distance"].mean(), 2),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("mIoU", ascending=False).reset_index(drop=True)
summary_df.to_csv("/kaggle/working/benchmark_summary.csv", index=False)

print("\n" + "="*75)
print(" FINAL BENCHMARK RESULTS")
print("="*75)
print(summary_df.to_string(index=False))
print("="*75)
print("Per-model CSVs and benchmark_summary.csv saved to /kaggle/working/")